# 01. Causal Discovery: NOTEARS Continuous Optimization, SCMs, and Instrumental Variables

Causal discovery and inference transcend purely correlational pattern recognition by constructing structural graphs that support intervention ($do$-calculus) and counterfactual query answering. This topic explores the continuous mathematical formulation of Directed Acyclic Graphs (DAGs) via NOTEARS, Pearl's Structural Causal Models (SCMs), and instrumental variable identification via Two-Stage Least Squares (2SLS).

### Core Causal Pillars
1. **Continuous DAG Optimization (NOTEARS)**: Transforms discrete graph structure search into smooth continuous optimization using the matrix exponential trace acyclicity constraint: $h(W) = \text{Tr}(e^{W \odot W}) - d = 0$.
2. **Structural Causal Models & Counterfactuals**: Implements Pearl's 3-step abduction-action-prediction counterfactual engine.
3. **Instrumental Variables (2SLS)**: Purges confounding bias between treatment and outcome by projecting onto an exogenous instrument subspace $P_Z = Z(Z^T Z)^{-1} Z^T$.


## Mathematical Foundations & Governing Equations

### 1. NOTEARS Acyclicity Constraint & Gradient
For weighted adjacency matrix $W \in \mathbb{R}^{d \times d}$:
$$h(W) = \text{Tr}\left( \exp(W \odot W) \right) - d = 0$$
$$\nabla h(W) = 2 \cdot \left( \exp(W \odot W) \right)^T \odot W$$

### 2. Pearl's 3-Step Counterfactual Engine
In a linear SCM $X = XW + U \iff X = U(I - W)^{-1}$:
- **Step 1 (Abduction)**: Infer factual exogenous noise: $u = x(I - W)$.
- **Step 2 (Action)**: Modify graph under $do(X_j = c)$: $W_{\text{int}}[:, j] = 0$, $u_{\text{int}}[j] = c$.
- **Step 3 (Prediction)**: Compute counterfactual: $x_{\text{cf}} = u_{\text{int}} (I - W_{\text{int}})^{-1}$.

### 3. Two-Stage Least Squares (2SLS)
- **Stage 1**: $\hat{\Gamma} = (Z^T Z)^{-1} Z^T T, \quad \hat{T} = Z \hat{\Gamma} = P_Z T$
- **Stage 2**: $\hat{\beta}_{2\text{SLS}} = (\hat{T}^T \hat{T})^{-1} \hat{T}^T Y$


In [ ]:
# Setup, imports, and golden weight loading
import sys
from pathlib import Path
import torch
from safetensors.torch import load_file

repo_root = Path.cwd()
while not (repo_root / "Cargo.toml").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

weights_path = repo_root / "weights" / "09_causal_ai_and_evolutionary_search" / "01_causal_discovery_notears_scm_and_instrumental_variables.safetensors"
assert weights_path.exists(), f"Weights file not found: {weights_path}"
weights = load_file(str(weights_path))
print(f"Loaded {len(weights)} golden tensors on cpu")


In [ ]:
def linear_anchor(x: torch.Tensor, weight: torch.Tensor, bias: torch.Tensor) -> torch.Tensor:
    """Canonical cross-track parity anchor."""
    return x @ weight + bias


def notears_acyclicity_constraint(w: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """NOTEARS matrix exponential acyclicity constraint and gradient."""
    d = w.shape[0]
    m = w * w
    exp_m = torch.matrix_exp(m)
    h = (torch.trace(exp_m) - float(d)).unsqueeze(0)
    grad_h = 2.0 * (exp_m.t() * w)
    return h, grad_h


def scm_generate_and_counterfactual(
    scm_w: torch.Tensor,
    scm_u: torch.Tensor,
    do_var_idx: int = 1,
    do_val: float = 3.0,
) -> tuple[torch.Tensor, torch.Tensor]:
    """SCM forward generation and counterfactual intervention."""
    d = scm_w.shape[0]
    eye = torch.eye(d, dtype=scm_w.dtype, device=scm_w.device)
    inv_w = torch.inverse(eye - scm_w)
    x_factual = scm_u @ inv_w

    w_intervened = scm_w.clone()
    w_intervened[:, do_var_idx] = 0.0
    u_intervened = scm_u.clone()
    u_intervened[0, do_var_idx] = do_val
    inv_w_int = torch.inverse(eye - w_intervened)
    x_counterfactual = u_intervened @ inv_w_int

    return x_factual, x_counterfactual


def two_stage_least_squares(
    z_inst: torch.Tensor,
    t_treat: torch.Tensor,
    y_outcome: torch.Tensor,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Two-Stage Least Squares (2SLS) Instrumental Variable Estimator."""
    zt_z = z_inst.t() @ z_inst
    zt_t = z_inst.t() @ t_treat
    gamma = torch.inverse(zt_z) @ zt_t
    t_hat = z_inst @ gamma

    th_th = t_hat.t() @ t_hat
    th_y = t_hat.t() @ y_outcome
    beta = torch.inverse(th_th) @ th_y

    return t_hat, beta


In [ ]:
# Multi-track numerical parity assertions
# 1. Linear anchor parity
y_anchor = linear_anchor(weights["test_input"], weights["weight"], weights["bias"])
err_anchor = float(torch.max(torch.abs(y_anchor - weights["test_output"])))
assert err_anchor < 1e-5, f"Anchor parity error: {err_anchor:.2e}"

# 2. NOTEARS acyclicity constraint
h_dag, grad_h = notears_acyclicity_constraint(weights["w_dag"])
err_h = float(torch.max(torch.abs(h_dag - weights["h_dag_ref"])))
assert err_h < 1e-5, f"NOTEARS h error: {err_h:.2e}"
err_grad = float(torch.max(torch.abs(grad_h - weights["grad_h_ref"])))
assert err_grad < 1e-5, f"NOTEARS grad error: {err_grad:.2e}"

# 3. SCM forward and counterfactual
x_fact, x_cf = scm_generate_and_counterfactual(weights["scm_w"], weights["scm_u"], do_var_idx=1, do_val=3.0)
err_fact = float(torch.max(torch.abs(x_fact - weights["scm_x_factual_ref"])))
assert err_fact < 1e-5, f"SCM factual error: {err_fact:.2e}"
err_cf = float(torch.max(torch.abs(x_cf - weights["counterfactual_x_ref"])))
assert err_cf < 1e-5, f"SCM counterfactual error: {err_cf:.2e}"

# 4. Two-Stage Least Squares (2SLS)
t_hat, beta_2sls = two_stage_least_squares(weights["z_inst"], weights["t_treat"], weights["y_outcome"])
err_that = float(torch.max(torch.abs(t_hat - weights["t_hat_ref"])))
assert err_that < 1e-5, f"2SLS T_hat error: {err_that:.2e}"
err_beta = float(torch.max(torch.abs(beta_2sls - weights["beta_2sls_ref"])))
assert err_beta < 1e-5, f"2SLS beta error: {err_beta:.2e}"

print(f"All parity assertions passed successfully (max error {max(err_anchor, err_h, err_grad, err_fact, err_cf, err_that, err_beta):.2e})")


In [ ]:
from core.benchmark import benchmark_fn, format_benchmark_table

b_notears = benchmark_fn(
    lambda: notears_acyclicity_constraint(weights["w_dag"]),
    iters=100,
    name="NOTEARS Acyclicity Constraint & Grad",
)
b_scm = benchmark_fn(
    lambda: scm_generate_and_counterfactual(weights["scm_w"], weights["scm_u"]),
    iters=100,
    name="SCM Forward & Counterfactual 3-Step",
)
b_2sls = benchmark_fn(
    lambda: two_stage_least_squares(weights["z_inst"], weights["t_treat"], weights["y_outcome"]),
    iters=100,
    name="Two-Stage Least Squares (2SLS)",
)

print(format_benchmark_table([b_notears, b_scm, b_2sls]))


In [ ]:
# Failure Mode Contrast: Naive OLS vs. Two-Stage Least Squares (2SLS) under Confounder Bias
# Ground truth data generating process: True causal effect beta = 2.50
# Unobserved confounder U affects both treatment T and outcome Y
torch.manual_seed(42)
z_sim = torch.randn(500, 2)            # Valid exogenous instrument (Z indep U)
u_sim = torch.randn(500, 1)            # Unobserved confounder
t_sim = z_sim @ torch.tensor([[1.5], [-0.8]]) + 1.2 * u_sim + torch.randn(500, 1) * 0.2
y_sim = 2.50 * t_sim + 2.0 * u_sim + torch.randn(500, 1) * 0.1

# Naive OLS estimation (Biased by Cov(T, U) != 0)
beta_ols = (torch.inverse(t_sim.t() @ t_sim) @ t_sim.t() @ y_sim).item()

# 2SLS Instrumental Variable estimation (Consistent!)
_, beta_2sls_sim = two_stage_least_squares(z_sim, t_sim, y_sim)
beta_2sls_val = beta_2sls_sim.item()

print(f"Ground Truth Causal Effect: 2.5000")
print(f"Naive OLS Estimate:         {beta_ols:.4f} (Severely biased by unobserved confounding!)")
print(f"2SLS Instrument Estimate:   {beta_2sls_val:.4f} (Consistent, recovers true causal effect!)")

assert abs(beta_ols - 2.50) > 0.05, f"Naive OLS should suffer from omitted variable bias"
assert abs(beta_2sls_val - 2.50) < 0.05, f"2SLS should accurately recover true causal effect"

print("Pathological failure mode contrast successfully verified")


## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Continuous Relaxation of Combinatorial Constraints**: The NOTEARS trace matrix exponential identity $h(W) = \text{Tr}(e^{W \odot W}) - d = 0$ converts super-exponential DAG search into smooth continuous non-linear optimization.
2. **Abduction-Action-Prediction**: Pearl's counterfactual engine isolates the idiosyncratic exogenous variables responsible for observed events before predicting alternative world histories under $do$-operations.
3. **Instrumental Purging**: Two-Stage Least Squares leverages exogenous variation $Z$ to isolate the unconfounded portion of treatment $T$, breaking endogeneity.

---
### Cross-Domain Generalisation Matrix
- **Genomic Gene Regulatory Networks**: NOTEARS discovers directed gene activation/inhibition pathways from perturbation scRNA-seq.
- **Econometrics & Policy Planning**: Instrumental variables isolate the causal return of education from unobserved innate ability.
- **Autonomous Driving & Robotics**: Counterfactual evaluation replays simulated sensor trajectories under modified braking actions.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [01_causal_discovery_notears_scm_and_instrumental_variables.md](../../markdown/09_causal_ai_and_evolutionary_search/01_causal_discovery_notears_scm_and_instrumental_variables.md)
- **PyTorch 2.x Module:** [01_causal_discovery_notears_scm_and_instrumental_variables.py](../../pytorch2/09_causal_ai_and_evolutionary_search/01_causal_discovery_notears_scm_and_instrumental_variables.py)
- **Functional JAX Module:** [01_causal_discovery_notears_scm_and_instrumental_variables.py](../../jax/09_causal_ai_and_evolutionary_search/01_causal_discovery_notears_scm_and_instrumental_variables.py)
- **Native Rust Candle Track:** [01_causal_discovery_notears_scm_and_instrumental_variables.rs](../../candle/09_causal_ai_and_evolutionary_search/01_causal_discovery_notears_scm_and_instrumental_variables.rs)
- **Hardware Kernel (CPU / WGSL):** [01_causal_discovery_notears_scm_and_instrumental_variables.rs](../../kernels/09_causal_ai_and_evolutionary_search/01_causal_discovery_notears_scm_and_instrumental_variables.rs)
- **Golden SafeTensors:** [01_causal_discovery_notears_scm_and_instrumental_variables.safetensors](../../weights/09_causal_ai_and_evolutionary_search/01_causal_discovery_notears_scm_and_instrumental_variables.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: Category 08 Topic 05: Test Time Compute PRM Best of N and MCTS](../08_learning_paradigms_rl_and_alignment/03_llm_alignment_and_reasoning/05_test_time_compute_prm_best_of_n_and_mcts.ipynb) | [Master Index](../../README.md#09_causal_ai_and_evolutionary_search) | [Next: 02. Causal Treatment Effects](02_causal_treatment_effects_tarnet_cfrnet_dragonnet_dml_and_r_learner.ipynb) &rarr;
